# Fine-Tuning Llama-3-8B for Schema-Guardian
## Using Unsloth + QLoRA for Binary Classification

This notebook outlines the strategy for fine-tuning Llama-3-8B to classify financial messages as **SAFE** or **FLAGGED**.

### 1. Data Preparation
Convert `dataset.jsonl` into the Alpaca format for training:
```python
def format_prompt(sample):
    return {
        "instruction": "Classify the following financial message as 'SAFE' or 'FLAGGED' based on semantic integrity.",
        "input": str(sample['message']),
        "output": sample['label']
    }
```

### 2. Fine-Tuning Strategy (Unsloth + QLoRA)
We use Unsloth for 2x faster training and 70% less memory usage.
```python
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
)
```

### 3. Hyperparameters
- **Batch Size:** 4
- **Gradient Accumulation:** 4
- **Epochs:** 1-3
- **Learning Rate:** 2e-4
- **Optimizer:** AdamW 8-bit

### 4. Evaluation Metrics
- **Binary Accuracy:** Correct classification (SAFE/FLAGGED).
- **Confusion Matrix:** Essential for identifying false negatives in financial data (i.e., missing a corrupted message).
- **Inference Latency:** Ensuring the fine-tuned adapter doesn't degrade performance.

### 5. Export to GGUF
After training, export the model for `llama.cpp` consumption:
```python
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
```